# Field Theory Framework — SageMath Demo

Model-agnostic MSRJD mean-field expansion using **SageMath**.  
Demonstrated on the **nonlinear Hawkes 2-population** model.

**Architecture**
- Field variables live in `PolynomialRing(SR, [ñ, ṽ, δṅ, δv])`
- Parameters, kernels, operators are SR expressions (polynomial coefficients)
- Bigrade $(n_\mathrm{tilde}, n_\mathrm{phys})$ read directly from exponent vectors via `.dict()`
- MF background conditions applied via `map_coefficients` on SR coefficients

### Sectors
| Bigrade $(n_\mathrm{tilde},\, n_\mathrm{phys})$ | Physical meaning |
|---|---|
| $(1,1)$ | Free action — determines propagators |
| $(\geq 2, 0)$ | Noise / source kernel |
| total $\geq 3$ | Interaction vertices |
| $(1,0)$, $(0,1)$ | Must vanish at MF saddle (sanity checks) |

In [ ]:
%display latex
from IPython.display import display, Math
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

load('field_theory_sage.py')
load('models/hawkes_sage.py')

print('Model:', HAWKES_MODEL['name'])
print('Convention:', HAWKES_MODEL['background_rate_convention'])

## 1. Build the polynomial ring and expand

`expand()` constructs the `PolynomialRing(SR, ring_vars)`, evaluates the action
lambda, substitutes the MF background conditions into SR coefficients, then
classifies every monomial by its exponent vector.

In [ ]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print('Polynomial ring:')
print(' ', R)
print()
print('Non-zero bigrade sectors:', sorted(ft.sectors().keys()))

## 2. Sanity checks

Three sectors must vanish:
- $(0,0)$ — constant: background action, no field dependence
- $(1,0)$ — tadpole: linear in $\tilde{n}_i, \tilde{v}_i$ only; vanishes by MF Poisson saddle $\dot{n}_i^* = -\varphi_i(v_i^*)$ and voltage EOM
- $(0,1)$ — EOM residual: linear in fluctuations only; vanishes at background solution

In [ ]:
passed = ft.sanity_check()

## 3. Full sector summary

In [ ]:
ft.summary()

## 4. Free action  $(n_\mathrm{tilde}=1,\, n_\mathrm{phys}=1)$

The quadratic kernel of the path integral — determines the Gaussian measure and propagators.

In [ ]:
S_free = ft.free_action()
print('Free action S_free:')
display(Math(latex(S_free)))

### Structural check: every monomial in the free action must have bigrade $(1,1)$

In [ ]:
n_tilde = ft._n_tilde
all_ok  = True
for exp_vec, coeff in S_free.dict().items():
    n_t = int(sum(exp_vec[:n_tilde]))
    n_p = int(sum(exp_vec[n_tilde:]))
    if (n_t, n_p) != (1, 1):
        print(f'  [FAIL] ({n_t},{n_p}): coeff={coeff}  exp={list(exp_vec)}')
        all_ok = False
if all_ok:
    print('[PASS] All free-action monomials have bigrade (1,1).')

## 5. Kernel matrix representation

Order the fields as
$$\tilde{\mathbf{a}} = [\tilde{v}_1,\, \tilde{v}_2,\, \tilde{n}_1,\, \tilde{n}_2]^\top,
\qquad \mathbf{a} = [\delta v_1,\, \delta v_2,\, \delta\dot{n}_1,\, \delta\dot{n}_2]^\top$$

The free action is $S_{\mathrm{free}} = \tilde{\mathbf{a}}^\top (\mathbf{K} \ast \mathbf{a})$.
The kernel matrix $\mathbf{K}$ is read directly from the $(1,1)$ polynomial coefficients.

Each entry $K_{ij}$ is the SR kernel (possibly involving $\partial_t$, $\delta$, coupling
constants) such that the $(i,j)$ bilinear coupling in $S_{\mathrm{free}}$ is
$\tilde{a}_i \cdot K_{ij} \ast a_j$.

In [ ]:
# Build position maps from actual ring generator order
ring_gen_names = [str(g) for g in R.gens()]

resp_names = [f'vt{i+1}' for i in ns.pop] + [f'nt{i+1}' for i in ns.pop]
phys_names = [f'dv{i+1}' for i in ns.pop] + [f'dn{i+1}' for i in ns.pop]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf     = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]

for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for k in range(len(ring_gen_names)):
        if exp_vec[k] > 0:
            if k in pos_to_row: row = pos_to_row[k]
            if k in pos_to_col: col = pos_to_col[k]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# ── Convert algebraic coefficients to kernel (distribution) form ─────────────
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

# ── Display K ─────────────────────────────────────────────────────────────────
resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]
resp_tex = (r'\begin{pmatrix}' + r' \\ '.join(latex(v) for v in resp_sr)
            + r'\end{pmatrix}')
phys_tex = (r'\begin{pmatrix}' + r' \\ '.join(latex(v) for v in phys_sr)
            + r'\end{pmatrix}')

display(Math(
    r'S_{\mathrm{free}} = \tilde{\mathbf{a}}^\top (\mathbf{K} \ast \mathbf{a})'
    r',\qquad \tilde{\mathbf{a}} = ' + resp_tex
    + r',\quad \mathbf{a} = ' + phys_tex))
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# ── Fourier transform of K (symbolic) ────────────────────────────────────────
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# ── Propagator: inverse of K̂  ────────────────────────────────────────────────
# Safeguards:
#   1. MAX_DIM  — skip symbolic inverse entirely for large systems
#   2. TIMEOUT  — kill .inverse() if it runs too long
#   3. MAX_NOPS — reject result if expressions are too bloated
import signal

MAX_DIM       = 20     # skip symbolic inverse above this matrix size
INVERSE_TIMEOUT = 60   # seconds before giving up on .inverse()
MAX_NOPS      = 5000   # max total operands across all G_ft entries

G_ft = None
G_ft_explicit = False

def _timeout_handler(signum, frame):
    raise TimeoutError('Symbolic matrix inverse exceeded time limit')

if nf > MAX_DIM:
    print(f'K̂ is {nf}x{nf} — skipping symbolic inverse (threshold: {MAX_DIM}x{MAX_DIM}).')
    print('Propagator will be stored implicitly via adjugate and determinant.')
    display(Math(r'\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)'
                 r'\quad\text{(stored implicitly)}'))
else:
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(INVERSE_TIMEOUT)
    try:
        G_ft_candidate = K_ft.inverse().apply_map(lambda e: e.factor())
        signal.alarm(0)  # cancel alarm on success
        total_nops = sum(G_ft_candidate[i, j].number_of_operands()
                         for i in range(nf) for j in range(nf)
                         if not G_ft_candidate[i, j].is_zero())
        if total_nops <= MAX_NOPS:
            G_ft = G_ft_candidate
            G_ft_explicit = True
            display(Math(r'\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega) = '
                         + latex(G_ft)))
        else:
            print(f'Symbolic inverse computed but too complex '
                  f'({total_nops} ops > {MAX_NOPS} threshold).')
            print('Storing K̂ and its adjugate/determinant instead.')
            display(Math(r'\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)'
                         r'\quad\text{(stored implicitly)}'))
    except (ZeroDivisionError, ValueError, RuntimeError, TimeoutError) as exc:
        signal.alarm(0)
        print(f'Symbolic matrix inverse failed: {exc}')
        print('Storing K̂ and its adjugate/determinant instead.')
        display(Math(r'\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)'
                     r'\quad\text{(stored implicitly)}'))
    finally:
        signal.signal(signal.SIGALRM, old_handler)

# Always compute adjugate and determinant — needed for residue theorem
# and usable even when the full inverse is too unwieldy
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()

if not G_ft_explicit:
    display(Math(r'\det\hat{\mathbf{K}}(\omega) = ' + latex(D_omega)))
    print(f'Adjugate and determinant stored as adj_ft, D_omega.')

# ── Inverse FT: branching logic ──────────────────────────────────────────────

if not D_omega.has(omega):
    # ── Branch 1: K̂ determinant is ω-independent ─────────────────────────────
    print('det(K̂) is independent of ω  →  G(t) = Ĝ · δ(t)')
    if G_ft_explicit:
        G_t = G_ft * dirac_delta(t_var)
        display(Math(r'\mathbf{G}(t) = \hat{\mathbf{G}}\,\delta(t) = '
                     + latex(G_ft) + r'\,\delta(t)'))
    else:
        G_t = None
        print('(G_ft not explicit — use adj_ft / D_omega · δ(t) at evaluation time)')

else:
    # det(K̂) depends on ω — look for poles
    D_prime = diff(D_omega, omega)

    pole_eqs = solve(D_omega == 0, omega)
    pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]
    n_poles  = len(pole_vals)

    if n_poles > 0:
        # ── Branch 2: poles found — residue theorem ──────────────────────────
        print(f'det(K̂) has {n_poles} pole(s) in ω  →  inverse FT via residue theorem')
        for k, p in enumerate(pole_vals):
            display(Math(r'\omega_{' + str(k+1) + '} = ' + latex(p)))

        C_mats = []
        for omega_k in pole_vals:
            C_data = [[SR(0)] * nf for _ in range(nf)]
            for i in range(nf):
                for j in range(nf):
                    n_ij = adj_ft[i, j]
                    if not n_ij.is_zero():
                        num = n_ij.subs({omega: omega_k})
                        den = D_prime.subs({omega: omega_k})
                        C_data[i][j] = (I * num / den).factor()
            C_mats.append(matrix(SR, C_data))

        print()
        for k, C in enumerate(C_mats):
            display(Math(r'\mathbf{C}_{' + str(k+1) + r'} = ' + latex(C)))

        omega_syms = ' + '.join(
            r'\mathbf{C}_{' + str(k+1) + r'}\, e^{i\omega_{' + str(k+1) + r'} t}'
            for k in range(n_poles))
        display(Math(r'\mathbf{G}(t) = ' + omega_syms
                     + r'\qquad (t > 0)'))

        G_t = sum(C_mats[k] * exp(I * pole_vals[k] * t_var)
                  for k in range(n_poles))
        G_t = G_t.apply_map(lambda e: e.simplify_full())

    else:
        # ── Branch 3: no poles found — try direct symbolic inverse FT ────────
        if G_ft_explicit:
            print('det(K̂) depends on ω but solve() found no poles')
            print('Attempting direct symbolic inverse FT...')

            G_t_data = [[SR(0)] * nf for _ in range(nf)]
            ift_ok = True
            for i in range(nf):
                for j in range(nf):
                    entry = G_ft[i, j]
                    if not entry.is_zero():
                        try:
                            result = inverse_fourier_transform(entry, omega, t_var)
                            if result.has(integrate):
                                ift_ok = False
                                break
                            G_t_data[i][j] = result.simplify_full()
                        except (ValueError, RuntimeError):
                            ift_ok = False
                            break
                if not ift_ok:
                    break

            if ift_ok:
                G_t = matrix(SR, G_t_data)
                print('Direct inverse FT succeeded.')
                display(Math(r'\mathbf{G}(t) = ' + latex(G_t)))
            else:
                G_t = None
                print('Inverse FT could not be computed symbolically.')
                print('Propagator stored in frequency domain as G_ft.')
                display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))
        else:
            # ── Branch 4: inverse too complex AND no poles — freq domain only
            G_t = None
            print('No poles found and G_ft too complex for entry-wise inverse FT.')
            print('Propagator available in frequency domain via adj_ft / D_omega.')

## 6. Noise kernel  $(n_\mathrm{tilde} \geq 2,\, n_\mathrm{phys} = 0)$

Sectors quadratic and higher in response fields — generate loop corrections to the propagator.

In [ ]:
nk = ft.noise_kernel()
print(f'Noise kernel sectors: {sorted(nk.keys())}')
for (n_t, n_p), expr in sorted(nk.items()):
    print(f'\n  (n_tilde={n_t}, n_phys={n_p}):')
    display(Math(latex(expr)))

## 7. Interaction vertices  (total degree $\geq 3$)

Feynman diagram insertions beyond the free-field level.

In [ ]:
verts = ft.vertices()
print(f'Vertex sectors: {sorted(verts.keys())}')
for (n_t, n_p), expr in sorted(verts.items()):
    print(f'\n  (n_tilde={n_t}, n_phys={n_p}):')
    display(Math(latex(expr)))

## 8. Free action in Conv / IP operator notation

Every bilinear coupling is $\mathrm{IP}(\tilde{\phi},\, \mathrm{Conv}(\kappa, \psi))$
with explicit kernels:

| Coupling | Kernel $\kappa$ |
|---|---|
| $\tilde{n}_i \cdot \delta\dot{n}_i$ | $\delta(t)$ |
| $\tilde{n}_i \cdot \delta v_i$ | $\varphi_i'\,\delta(t)$ |
| $\tilde{v}_i \cdot \delta v_i$ | $\tau\delta'(t) + \delta(t)$ |
| $\tilde{v}_i \cdot \delta\dot{n}_j$ | $w_{ij}\,g(t)$ |

In [ ]:
pop      = ns.pop
phi1s    = [SR.var(f'phi1_{i+1}') for i in pop]
delta_D  = ns.delta_D    # delta(t)
delta_Dp = ns.delta_Dp   # delta'(t)
tau      = ns.tau

terms = []
for i in pop:
    terms.append(  IP(ns.nt[i],  Conv(delta_D,                   ns.dn[i])))
    terms.append(  IP(ns.nt[i],  Conv(phi1s[i] * delta_D,        ns.dv[i])))
    terms.append(  IP(ns.vt[i],  Conv(tau * delta_Dp + delta_D,  ns.dv[i])))
    for j in pop:
        terms.append(-IP(ns.vt[i], Conv(ns.w[i][j] * ns.g,       ns.dn[j])))

S_free_conv = _DisplaySum(terms)

print('Free action in Conv/IP notation:')
display(Math(latex(S_free_conv)))

## 9. Inspect the raw polynomial ring

Access the underlying polynomial data directly — useful for further algebraic manipulation.

In [ ]:
print('Ring generators (tilde first, then physical):')
print(' ', R.gens())
print()
print('n_tilde generators:', ft._n_tilde)
print()
print('(2,0) noise kernel term — monomial dict:')
nk20 = ft.noise_kernel().get((2,0), R.zero())
for exp_vec, coeff in nk20.dict().items():
    print(f'  exp={list(exp_vec)}  coeff={coeff}')

---
## Extending to a new model

Define a new model dict with the same keys and pass it to `FieldTheory`:

```python
MY_MODEL = {
    'name': 'My stochastic model',
    'index_sets':       {'pop': [0]},
    'response_fields':  [{'name': 'nt', 'indexed': True}],
    'physical_fields':  [{'name': 'dn', 'indexed': True}],
    'parameters':       [{'name': 'mu', 'indexed': False}],
    'kernels':          [],
    'operators':        [],
    'functions':        [...],
    'mf_substitutions': [...],
    'mf_bg_conditions': lambda ns: {...},
    'background_rate_convention': '...',
    'action': lambda ns: ...
}

ft2 = FieldTheory(MY_MODEL, taylor_order=4)
ft2.expand()
ft2.sanity_check()
ft2.summary()
```

See `models/hawkes_sage.py` for the complete format reference.